# Recall figures

Renders the per-transformation recall bar grids (**Figures 6-7**) from the JSON
produced by `fpr_recall_evaluation.ipynb`. Set `OUTPUT_DIR` to the folder that
holds `recall_domainnet.json` / `recall_negative_dataset.json`.


In [ ]:
import os
OUTPUT_DIR = "metrics"  # folder containing the recall_*.json files


In [ ]:
import json
from pathlib import Path
from typing import Optional, Union, List, Dict

import numpy as np
import matplotlib.pyplot as plt

# Порядок моделей как в таблицах статьи (legend и столбики)
DEFAULT_MODEL_ORDER = [
    "Ours-ViT-pre",
    "Ours-ViT-sft",
    "Ours-EffNet-pre",
    "Ours-EffNet-sft",
    "SiamNet-pre",
    "SiamNet-sft",
    "Qwen3-VL-4B-Instruct",
]


def plot_recall_robustness_from_json(
    results_path: Union[str, Path],
    aug_names: Optional[List[str]] = None,
    exclude_categories: Optional[List[str]] = None,
    model_order: Optional[List[str]] = None,
    figsize: tuple = (18, 12),
    output_path: Optional[Union[str, Path]] = None,
) -> None:

    results_path = Path(results_path)

    with results_path.open("r", encoding="utf-8") as f:
        raw_data: Dict = json.load(f)

    results_dict: Dict[str, Dict[str, Dict[str, float]]] = {
        model_name: entry["results"]
        for model_name, entry in raw_data.items()
        if isinstance(entry, dict) and "results" in entry
    }

    if not results_dict:
        raise ValueError(f"No valid 'results' found in input data at {results_path}")

    order = model_order if model_order is not None else DEFAULT_MODEL_ORDER
    models = [m for m in order if m in results_dict]
    models += sorted(m for m in results_dict.keys() if m not in models)

    exclude = set(exclude_categories or [])
    categories = sorted({
        cat
        for model_res in results_dict.values()
        for cat in model_res.keys()
        if cat not in exclude
    })

    if aug_names is None:
        all_augs = {
            aug
            for model_res in results_dict.values()
            for cat_res in model_res.values()
            for aug in cat_res.keys()
        }
        aug_names = sorted(all_augs)

    def _aug_order_key(a: str) -> tuple:
        parts = a.split("_")
        is_combined = 1 if len(parts) > 2 else 0
        return (is_combined, a)

    aug_names = sorted(aug_names, key=_aug_order_key)
    aug_names = aug_names[:9]

    n_rows, n_cols = 3, 3
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    axes = np.array(axes).reshape(n_rows, n_cols)

    colors = [plt.cm.tab10(i % 10) for i in range(len(models))]
    x_pos = np.arange(len(categories))
    bar_width = 0.8 / max(len(models), 1)

    for idx, aug_name in enumerate(aug_names):
        r, c = divmod(idx, n_cols)
        ax = axes[r, c]

        recalls_per_model: Dict[str, List[float]] = {m: [] for m in models}
        for cat in categories:
            for model in models:
                recall_val = results_dict[model].get(cat, {}).get(aug_name, 0.0)
                recalls_per_model[model].append(recall_val)

        for i, model in enumerate(models):
            ax.bar(
                x_pos + i * bar_width,
                recalls_per_model[model],
                width=bar_width,
                label=model if idx == 0 else "",
                color=colors[i],
                edgecolor="white",
                linewidth=0.5,
            )

        ax.set_xticks(x_pos + bar_width * (len(models) - 1) / 2)
        ax.set_xticklabels(categories, fontsize=9, rotation=30, ha="right")
        ax.set_ylim(0.0, 1.02)
        ax.set_ylabel("Recall", fontsize=10)
        ax.set_title(aug_name.replace("_", " ").title(), fontsize=12, pad=8)
        ax.grid(axis="y", linestyle="--", alpha=0.6)
        ax.tick_params(axis="y", labelsize=8)

    for idx in range(len(aug_names), n_rows * n_cols):
        r, c = divmod(idx, n_cols)
        axes[r, c].axis("off")

    handles = [
        plt.Rectangle((0, 0), 1, 1, color=colors[i], label=model)
        for i, model in enumerate(models)
    ]
    fig.legend(
        handles,
        models,
        loc="lower center",
        ncol=min(len(models), 4),
        fontsize=12,
    )

    plt.tight_layout(rect=[0, 0.06, 1, 0.96])

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Grid saved to: {output_path}")

    plt.show()

In [ ]:
plt.rcParams.update({
    'text.usetex': False,
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif'],
    'font.size': 14,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'lines.linewidth': 1.5,
    'lines.markersize': 8,
    'grid.alpha': 0.3,
    'figure.dpi': 300,
    'font.weight': 'normal',
})

In [ ]:
plot_recall_robustness_from_json(
    OUTPUT_DIR + "/recall_domainnet.json",
    output_path = OUTPUT_DIR + "/recall_domainnet.pdf"
)

# Для curated negative dataset (твои категории: diagram, face, ... , text)
# plot_recall_robustness_from_json(
#     OUTPUT_DIR + "/recall_negative_dataset.json",
#     exclude_categories=['monotone', 'other'],
#     output_path = OUTPUT_DIR + "/recall_negative_dataset.pdf"
# )

In [ ]:
import json
from pathlib import Path

def merge_json(
    base_path: str,
    extra_path: str,
    output_path: str,
) -> None:
    base_path = Path(base_path)
    extra_path = Path(extra_path)
    output_path = Path(output_path)

    with base_path.open("r", encoding="utf-8") as f:
        base = json.load(f)

    with extra_path.open("r", encoding="utf-8") as f:
        extra = json.load(f)

    # добавляем / перезаписываем все ключи из extra в base
    if not isinstance(base, dict) or not isinstance(extra, dict):
        raise ValueError("Оба JSON файла должны содержать объект (dict) на верхнем уровне.")

    base.update(extra)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(base, f, indent=2, ensure_ascii=False)

    print(f"Merged JSON saved to: {output_path}")